In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)  # fixed seed for reproducibility

# load the pre-filtered WYCA disruptions (April-June 2026 window) from notebook 01
scoped_disruptions = pd.read_csv(
    "../data/interim/wyca_disruptions_scoped.csv",
    parse_dates=['Validity start', 'Validity end']
)
scoped_disruptions['Validity start'] = scoped_disruptions['Validity start'].dt.tz_convert('UTC')
scoped_disruptions['Validity end'] = scoped_disruptions['Validity end'].dt.tz_convert('UTC')

window_start = pd.Timestamp("2026-04-01", tz="UTC")
window_end = pd.Timestamp("2026-06-30 23:59:59", tz="UTC")

# --- compute duration and severity ---
scoped_disruptions['is_ongoing'] = scoped_disruptions['Validity end'].isna()
scoped_disruptions['duration_minutes'] = (
    scoped_disruptions['Validity end'] - scoped_disruptions['Validity start']
).dt.total_seconds() / 60

def severity_label(row):
    # no end date means the disruption was still active at time of data pull
    if row['is_ongoing']:
        return 'ongoing'
    elif row['duration_minutes'] <= 240:
        return 'minor'
    elif row['duration_minutes'] <= 1440:
        return 'moderate'
    else:
        return 'severe'

scoped_disruptions['severity'] = scoped_disruptions.apply(severity_label, axis=1)

# clip disruption validity to the 3-month analysis window so overlap makes sense
# even for disruptions that started earlier or are still ongoing
scoped_disruptions['overlap_start'] = scoped_disruptions['Validity start'].clip(lower=window_start)
scoped_disruptions['overlap_end'] = scoped_disruptions['Validity end'].fillna(window_end).clip(upper=window_end)

print(scoped_disruptions.shape)
scoped_disruptions[['ID', 'Reason', 'is_ongoing', 'duration_minutes', 'severity', 'Services affected', 'overlap_start', 'overlap_end']]

(24, 17)


,ID,Reason,is_ongoing,duration_minutes,severity,Services affected,overlap_start,overlap_end
0,2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587,roadworks,True,NaN,ongoing,1.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
1,de178bce-11a5-4a61-9897-5360b40c0e37,roadClosed,False,537805.0,severe,9.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
2,289217e6-6830-4718-97f9-59621a5064f2,roadworks,False,1027799.0,severe,8.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
3,f258b9b9-0c10-4e73-b4f9-830781431919,roadworks,True,NaN,ongoing,1.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
4,630d9779-55de-4b37-9a2c-fe0c70285bb1,roadworks,True,NaN,ongoing,11.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
5,706b6b5f-532e-4e92-ba5b-80418e2c5807,roadClosed,False,266992.0,severe,1.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
6,7b4e17d3-95c1-46aa-99c0-08ff6e350bfd,roadworks,True,NaN,ongoing,12.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
7,f9b01a78-5ff0-4e37-a4a7-2b2d65a98261,routeDiversion,False,399959.0,severe,1.0,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
8,43864a74-527d-494b-941a-dc2c99b2210e,roadworks,False,307431.0,severe,1.0,2026-04-02 12:08:00+00:00,2026-06-30 23:59:59+00:00
9,fce0f123-f591-46ef-9242-d970c24d1b85,roadworks,False,408659.0,severe,2.0,2026-04-07 05:00:00+00:00,2026-06-30 23:59:59+00:00


**duration measured within our 3-month analysis window, not the full validity span,** 

**since validity end dates often reflect a permit's max allowed duration rather than how long the disruption practically affected service**

In [2]:
scoped_disruptions['overlap_duration_minutes'] = (
    scoped_disruptions['overlap_end'] - scoped_disruptions['overlap_start']
).dt.total_seconds() / 60

def severity_label(row):
    d = row['overlap_duration_minutes']
    if d <= 1440:          # up to 1 day
        return 'minor'
    elif d <= 10080:       # up to 1 week
        return 'moderate'
    else:
        return 'severe'

scoped_disruptions['severity'] = scoped_disruptions.apply(severity_label, axis=1)

print(scoped_disruptions['severity'].value_counts())
scoped_disruptions[['ID', 'Reason', 'is_ongoing', 'overlap_duration_minutes', 'severity', 'Services affected']]

severity
severe    24
Name: count, dtype: int64


,ID,Reason,is_ongoing,overlap_duration_minutes,severity,Services affected
0,2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587,roadworks,True,131039.983333,severe,1.0
1,de178bce-11a5-4a61-9897-5360b40c0e37,roadClosed,False,131039.983333,severe,9.0
2,289217e6-6830-4718-97f9-59621a5064f2,roadworks,False,131039.983333,severe,8.0
3,f258b9b9-0c10-4e73-b4f9-830781431919,roadworks,True,131039.983333,severe,1.0
4,630d9779-55de-4b37-9a2c-fe0c70285bb1,roadworks,True,131039.983333,severe,11.0
5,706b6b5f-532e-4e92-ba5b-80418e2c5807,roadClosed,False,131039.983333,severe,1.0
6,7b4e17d3-95c1-46aa-99c0-08ff6e350bfd,roadworks,True,131039.983333,severe,12.0
7,f9b01a78-5ff0-4e37-a4a7-2b2d65a98261,routeDiversion,False,131039.983333,severe,1.0
8,43864a74-527d-494b-941a-dc2c99b2210e,roadworks,False,128871.983333,severe,1.0
9,fce0f123-f591-46ef-9242-d970c24d1b85,roadworks,False,122099.983333,severe,2.0


**This drops the old duration_minutes/is_ongoing-as-category approach in favor of one consistent, bounded, meaningful metric.** 

**Using quantile-based thresholds instead of fixed cutoffs,**
    
**since fixed guesses didn't match this dataset's actual spread (all values landed in one bucket**

In [3]:
q33 = scoped_disruptions['overlap_duration_minutes'].quantile(0.33)
q66 = scoped_disruptions['overlap_duration_minutes'].quantile(0.66)

print(f"33rd percentile: {q33:.0f} min ({q33/1440:.1f} days)")
print(f"66th percentile: {q66:.0f} min ({q66/1440:.1f} days)")

def severity_label(row):
    d = row['overlap_duration_minutes']
    if d <= q33:
        return 'minor'
    elif d <= q66:
        return 'moderate'
    else:
        return 'severe'

scoped_disruptions['severity'] = scoped_disruptions.apply(severity_label, axis=1)

print(scoped_disruptions['severity'].value_counts())
scoped_disruptions[['ID', 'Reason', 'overlap_duration_minutes', 'severity', 'Services affected']].sort_values('overlap_duration_minutes')

33rd percentile: 63700 min (44.2 days)
66th percentile: 129262 min (89.8 days)
severity
severe      8
moderate    8
minor       8
Name: count, dtype: int64


,ID,Reason,overlap_duration_minutes,severity,Services affected
23,d67e5b21-f0dc-4143-a6a4-de4ddf9090c7,roadworks,10874.983333,minor,6.0
22,b7921958-134b-48e9-9913-4ed5ad3e50ce,roadworks,21899.983333,minor,2.0
21,5a57b8f5-222e-4e54-951e-fe0508019f9a,roadworks,23998.983333,minor,1.0
20,7f89b42c-8814-4873-8232-5e39c0642d96,roadworks,32542.983333,minor,1.0
19,b1fc65da-5ac6-4d75-9604-5f40c88d7b51,roadworks,32819.983333,minor,2.0
18,3ea82274-0db3-441a-a725-1a41e238be59,roadworks,41230.983333,minor,1.0
17,aeeab4e6-ae1e-4dec-ae8c-f5136bcc62dd,roadworks,58632.983333,minor,2.0
16,c45f870c-2272-403b-9d9c-9d3eab5debda,roadworks,58739.983333,minor,3.0
15,9f0ba969-9631-4311-9956-a06ff6a41673,roadworks,67146.983333,moderate,3.0
14,abdfb8c1-7f5e-4396-9140-d427070bda64,roadworks,70059.983333,moderate,1.0


# Sampling Join :-

In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("DisruptionJoin") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# load the expanded trip instances built in notebook 02
trip_instances = spark.read.parquet("../data/interim/trip_instances_expanded")
print("Total trip instances:", trip_instances.count())

# pull distinct (operator, line_name) pairs — this is the sampling pool,
# since disruptions only tell us a COUNT of services affected, not which ones
distinct_lines = trip_instances.select("operator_name", "line_name").distinct().toPandas()
print("Distinct operator/line combinations:", len(distinct_lines))
distinct_lines.head(10)

26/07/14 10:22:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Total trip instances: 765089
Distinct operator/line combinations: 195


,operator_name,line_name
0,Arriva Yorkshire,280
1,Arriva Yorkshire,283
2,Arriva Yorkshire,493
3,Arriva Yorkshire,261
4,Arriva Yorkshire,425
5,Arriva Yorkshire,164
6,Arriva Yorkshire,141
7,Arriva Yorkshire,202
8,Arriva Yorkshire,230A
9,FIRST WEST YORKSHIRE LTD,590


In [5]:
import numpy as np

np.random.seed(42)  # reproducibility

# build the disruption -> sampled lines mapping
mapping_rows = []

for _, disruption in scoped_disruptions.iterrows():
    n_services = int(round(disruption['Services affected']))
    n_services = min(n_services, len(distinct_lines))  # safety cap, just in case

    sampled = distinct_lines.sample(n=n_services, random_state=42 + _)  # varying seed per row so each disruption samples differently

    for _, line in sampled.iterrows():
        mapping_rows.append({
            'disruption_id': disruption['ID'],
            'operator_name': line['operator_name'],
            'line_name': line['line_name'],
            'reason': disruption['Reason'],
            'planned': disruption['Planned'],
            'severity': disruption['severity'],
            'overlap_duration_minutes': disruption['overlap_duration_minutes'],
            'overlap_start': disruption['overlap_start'],
            'overlap_end': disruption['overlap_end'],
        })

disruption_line_map = pd.DataFrame(mapping_rows)
print("Disruption-line mapping rows:", len(disruption_line_map))
disruption_line_map.head(10)

Disruption-line mapping rows: 83


,disruption_id,operator_name,line_name,reason,planned,severity,overlap_duration_minutes,overlap_start,overlap_end
0,2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587,The Harrogate Bus Company,36,roadworks,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
1,de178bce-11a5-4a61-9897-5360b40c0e37,Arriva Yorkshire,203,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
2,de178bce-11a5-4a61-9897-5360b40c0e37,The Keighley Bus Company,864,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
3,de178bce-11a5-4a61-9897-5360b40c0e37,Team Pennine,C35,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
4,de178bce-11a5-4a61-9897-5360b40c0e37,The Keighley Bus Company,62,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
5,de178bce-11a5-4a61-9897-5360b40c0e37,The Keighley Bus Company,66,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
6,de178bce-11a5-4a61-9897-5360b40c0e37,Arriva Yorkshire,174,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
7,de178bce-11a5-4a61-9897-5360b40c0e37,First Leeds,28,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
8,de178bce-11a5-4a61-9897-5360b40c0e37,First Bradford,72,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00
9,de178bce-11a5-4a61-9897-5360b40c0e37,Arriva Yorkshire,104,roadClosed,True,severe,131039.983333,2026-04-01 00:00:00+00:00,2026-06-30 23:59:59+00:00


In [6]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, BooleanType

# Spark's TimestampType has no timezone concept of its own — it silently applies
# the local system timezone on conversion from tz-aware pandas columns.
# Strip tz info here (values are already UTC) so both overlap_start/overlap_end
# and scheduled_datetime represent the same naive wall-clock time.
disruption_line_map['overlap_start'] = disruption_line_map['overlap_start'].dt.tz_localize(None)
disruption_line_map['overlap_end'] = disruption_line_map['overlap_end'].dt.tz_localize(None)

# converting the pandas mapping into a Spark DataFrame so it can join against trip_instances
disruption_line_map_spark = spark.createDataFrame(disruption_line_map)
disruption_line_map_spark = disruption_line_map_spark.withColumn(
    "overlap_start", F.col("overlap_start").cast(TimestampType())
).withColumn(
    "overlap_end", F.col("overlap_end").cast(TimestampType())
)

disruption_line_map_spark.printSchema()
disruption_line_map_spark.show(5)

root
 |-- disruption_id: string (nullable = true)
 |-- operator_name: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- reason: string (nullable = true)
 |-- planned: boolean (nullable = true)
 |-- severity: string (nullable = true)
 |-- overlap_duration_minutes: double (nullable = true)
 |-- overlap_start: timestamp (nullable = true)
 |-- overlap_end: timestamp (nullable = true)



[Stage 7:>                                                          (0 + 1) / 1]

+--------------------+--------------------+---------+----------+-------+--------+------------------------+-------------------+-------------------+
|       disruption_id|       operator_name|line_name|    reason|planned|severity|overlap_duration_minutes|      overlap_start|        overlap_end|
+--------------------+--------------------+---------+----------+-------+--------+------------------------+-------------------+-------------------+
|2b9e1d8f-b0ee-43a...|The Harrogate Bus...|       36| roadworks|   true|  severe|      131039.98333333334|2026-04-01 00:00:00|2026-06-30 23:59:59|
|de178bce-11a5-4a6...|    Arriva Yorkshire|      203|roadClosed|   true|  severe|      131039.98333333334|2026-04-01 00:00:00|2026-06-30 23:59:59|
|de178bce-11a5-4a6...|The Keighley Bus ...|      864|roadClosed|   true|  severe|      131039.98333333334|2026-04-01 00:00:00|2026-06-30 23:59:59|
|de178bce-11a5-4a6...|        Team Pennine|      C35|roadClosed|   true|  severe|      131039.98333333334|2026-04-01 0

In [7]:
# join trip_instances to the disruption mapping on operator + line name,
# then keep only trips whose scheduled time actually falls inside
# that specific disruption's overlap window
disrupted_trips = trip_instances.join(
    disruption_line_map_spark,
    on=["operator_name", "line_name"],
    how="inner"
).filter(
    (F.col("scheduled_datetime") >= F.col("overlap_start")) &
    (F.col("scheduled_datetime") <= F.col("overlap_end"))
)

disrupted_trips = disrupted_trips.withColumn("is_disrupted", F.lit(1))
disrupted_trips.cache()
print("Disrupted trip rows:", disrupted_trips.count())

26/07/14 10:22:13 WARN MemoryStore: Not enough space to cache broadcast_10 in memory! (computed 328.0 MiB so far)
26/07/14 10:22:13 WARN BlockManager: Persisting block broadcast_10 to disk instead.
26/07/14 10:22:14 WARN MemoryStore: Not enough space to cache broadcast_10 in memory! (computed 296.0 MiB so far)
26/07/14 10:22:15 WARN MemoryStore: Not enough space to cache broadcast_10 in memory! (computed 296.0 MiB so far)
[Stage 9:====================================>                      (5 + 3) / 8]

Disrupted trip rows: 227163


In [8]:
from pyspark.sql import Window

# a trip could theoretically be sampled into two overlapping disruptions —
# keep only the more severe/longer one per trip so each trip maps to exactly one disruption
severity_rank = F.when(F.col("severity") == "severe", 3) \
                  .when(F.col("severity") == "moderate", 2) \
                  .otherwise(1)

disrupted_trips = disrupted_trips.withColumn("severity_rank", severity_rank)

window_spec = Window.partitionBy(
    "operator_name", "line_name", "trip_date", "departure_time", "service_code"
).orderBy(F.col("severity_rank").desc(), F.col("overlap_duration_minutes").desc())

disrupted_trips = disrupted_trips.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num", "severity_rank")

disrupted_trips.cache()
print("Disrupted trip rows (deduplicated):", disrupted_trips.count())

[Stage 15:>                                                         (0 + 8) / 8]

Disrupted trip rows (deduplicated): 180949


In [9]:
# trips that were never matched to any disruption become the baseline "normal" class
disrupted_trip_keys = disrupted_trips.select("operator_name", "line_name", "trip_date", "departure_time", "service_code").distinct()

normal_trips = trip_instances.join(
    disrupted_trip_keys,
    on=["operator_name", "line_name", "trip_date", "departure_time", "service_code"],
    how="left_anti"  # keep only trips with NO match in disrupted set
)

normal_trips = normal_trips.withColumn("is_disrupted", F.lit(0)) \
    .withColumn("reason", F.lit(None).cast("string")) \
    .withColumn("planned", F.lit(None).cast("boolean")) \
    .withColumn("severity", F.lit("none")) \
    .withColumn("overlap_duration_minutes", F.lit(None).cast("double"))

print("Normal (undisrupted) trip rows:", normal_trips.count())

[Stage 24:>                                                         (0 + 8) / 8]

Normal (undisrupted) trip rows: 584140


In [10]:
# diagnostic: checking whether trip_instances has duplicate rows sharing the same
# key columns — if so, the anti-join would remove ALL duplicates for any
# matched key, not just one, which would explain missing rows
key_cols = ["operator_name", "line_name", "trip_date", "departure_time", "service_code"]

total_rows = trip_instances.count()
distinct_key_rows = trip_instances.select(*key_cols).distinct().count()

print("Total trip_instances rows:", total_rows)
print("Distinct key combinations:", distinct_key_rows)
print("Duplicate rows (same key appearing more than once):", total_rows - distinct_key_rows)

disrupted_key_count = disrupted_trips.select(*key_cols).distinct().count()
print("\nDisrupted trips row count:", disrupted_trips.count())
print("Disrupted trips distinct key count:", disrupted_key_count)

Total trip_instances rows: 765089
Distinct key combinations: 765089
Duplicate rows (same key appearing more than once): 0

Disrupted trips row count: 180949
Disrupted trips distinct key count: 180949


In [11]:
# combine disrupted and normal trips into one unified, model-ready dataset
final_columns = [
    "operator_name", "line_name", "origin", "destination", "trip_date",
    "weekday_name", "departure_time", "scheduled_datetime", "service_code",
    "is_disrupted", "reason", "planned", "severity", "overlap_duration_minutes"
]

disrupted_final = disrupted_trips.select(*final_columns)
normal_final = normal_trips.select(*final_columns)

final_dataset = disrupted_final.unionByName(normal_final)
final_dataset.cache()

print("Final dataset rows:", final_dataset.count())
print("is_disrupted distribution:")
final_dataset.groupBy("is_disrupted").count().show()

Final dataset rows: 765089
is_disrupted distribution:
+------------+------+
|is_disrupted| count|
+------------+------+
|           1|180949|
|           0|584140|
+------------+------+



In [12]:
# save the final joined, model-ready dataset
final_dataset.write.mode("overwrite").parquet("../data/processed/trips_with_disruptions")
print("Saved to data/processed/trips_with_disruptions")

26/07/14 10:33:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 68:>                                                         (0 + 8) / 8]

Saved to data/processed/trips_with_disruptions


In [13]:
final_dataset.select(
    "operator_name", "line_name", "trip_date", "departure_time",
    "is_disrupted", "reason", "severity", "overlap_duration_minutes"
).orderBy(F.rand(seed=1)).show(10, truncate=False)

+------------------------+---------+----------+--------------+------------+---------+--------+------------------------+
|operator_name           |line_name|trip_date |departure_time|is_disrupted|reason   |severity|overlap_duration_minutes|
+------------------------+---------+----------+--------------+------------+---------+--------+------------------------+
|Go Ahead West Yorkshire |633      |2026-04-06|11:55:00      |0           |NULL     |none    |NULL                    |
|The Keighley Bus Company|662      |2026-04-04|17:05:00      |0           |NULL     |none    |NULL                    |
|Arriva Yorkshire        |165      |2026-06-16|09:20:00      |0           |NULL     |none    |NULL                    |
|The Keighley Bus Company|K9       |2026-06-03|15:32:00      |0           |NULL     |none    |NULL                    |
|The Keighley Bus Company|B1       |2026-06-05|20:35:00      |0           |NULL     |none    |NULL                    |
|First Leeds             |23       |2026